# 02. Подготовка обучающих данных

Ноутбук делает две вещи:

1. превращает синие экспертные контуры в предварительные маски областей оталькования;
2. формирует очищенный manifest для классификатора `ordinary / fine`, удаляя дубли и противоречивые метки.

Исходные изображения не изменяются.

Перед запуском сначала должен быть выполнен `01_data_audit.ipynb`.

In [1]:
from pathlib import Path

AUDIT_DIR = Path("outputs/audit")
PROCESSED_DIR = Path("data/processed")

TALC_MASK_DIR = PROCESSED_DIR / "talc_masks"
TALC_OVERLAY_DIR = PROCESSED_DIR / "talc_overlays"
REVIEW_DIR = PROCESSED_DIR / "talc_review"
MANIFEST_DIR = PROCESSED_DIR / "manifests"

for folder in [
    TALC_MASK_DIR,
    TALC_OVERLAY_DIR,
    REVIEW_DIR,
    MANIFEST_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

required_files = [
    AUDIT_DIR / "dataset_inventory.csv",
    AUDIT_DIR / "talc_pairs.csv",
]

for file in required_files:
    assert file.exists(), (
        f"Не найден файл: {file}\n"
        "Сначала выполните 01_data_audit.ipynb."
    )

print("AUDIT_DIR:", AUDIT_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())

AUDIT_DIR: C:\Users\Мария\ore_hackathon\outputs\audit
PROCESSED_DIR: C:\Users\Мария\ore_hackathon\data\processed


In [2]:
import hashlib
import math
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)
Image.MAX_IMAGE_PIXELS = None


def load_rgb(path: str | Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(
            ImageOps.exif_transpose(image).convert("RGB")
        )


def save_mask(mask: np.ndarray, path: Path) -> None:
    Image.fromarray((mask > 0).astype(np.uint8) * 255).save(path)


def make_overlay(
    rgb: np.ndarray,
    mask: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:
    result = rgb.astype(np.float32).copy()
    blue = np.zeros_like(result)
    blue[..., 2] = 255

    selected = mask.astype(bool)
    result[selected] = (
        (1 - alpha) * result[selected]
        + alpha * blue[selected]
    )
    return np.clip(result, 0, 255).astype(np.uint8)

## 1. Проверка результатов аудита

In [3]:
inventory = pd.read_csv(
    AUDIT_DIR / "dataset_inventory.csv",
    encoding="utf-8-sig",
)

pairs = pd.read_csv(
    AUDIT_DIR / "talc_pairs.csv",
    encoding="utf-8-sig",
)

print("Всего изображений:", len(inventory))
display(
    inventory["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

print("Сопоставленных пар талька:",
      int((pairs["status"] == "matched").sum()))
print("Пар одинакового размера:",
      int(pairs["shape_equal"].fillna(False).sum()))

Всего изображений: 1236


,label,count
0,ordinary,565
1,fine,486
2,talc,129
3,talc_annotated,42
4,panorama,14


Сопоставленных пар талька: 42
Пар одинакового размера: 42


## 2. Извлечение синей линии

Порог учитывает JPEG-артефакты вокруг линии.

In [4]:
def extract_blue_line(rgb: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    lower_blue = np.array([95, 70, 45], dtype=np.uint8)
    upper_blue = np.array([145, 255, 255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Закрываем небольшие разрывы после JPEG-сжатия.
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2,
    )

    # Удаляем единичный цветной шум.
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        (mask > 0).astype(np.uint8),
        connectivity=8,
    )

    cleaned = np.zeros_like(mask)
    for component_id in range(1, count):
        area = stats[component_id, cv2.CC_STAT_AREA]
        if area >= 20:
            cleaned[labels == component_id] = 255

    return cleaned

## 3. Превращение контуров в маски

Логика:

- синяя линия становится непроходимой границей;
- если линия заканчивается рядом с краем, она продолжается до края;
- рамка изображения также считается границей;
- изображение делится на связные области;
- крупнейшая область принимается за внешнюю, неразмеченную;
- остальные достаточно крупные области считаются отмеченными экспертом.

Это **псевдомаски**, а не окончательная истина. Поэтому далее автоматически создаются страницы визуальной проверки.

In [5]:
def connect_line_to_border(
    line_mask: np.ndarray,
    margin: int = 24,
) -> np.ndarray:
    """
    Продлевает части синей линии, расположенные близко к краю,
    до рамки изображения.
    """
    result = line_mask.copy()
    h, w = result.shape

    ys, xs = np.where(result > 0)

    if len(xs) == 0:
        return result

    # Левая граница.
    near = np.where(xs < margin)[0]
    for idx in near[::max(1, len(near) // 50 or 1)]:
        cv2.line(result, (0, int(ys[idx])), (int(xs[idx]), int(ys[idx])), 255, 3)

    # Правая граница.
    near = np.where(xs >= w - margin)[0]
    for idx in near[::max(1, len(near) // 50 or 1)]:
        cv2.line(result, (int(xs[idx]), int(ys[idx])), (w - 1, int(ys[idx])), 255, 3)

    # Верхняя граница.
    near = np.where(ys < margin)[0]
    for idx in near[::max(1, len(near) // 50 or 1)]:
        cv2.line(result, (int(xs[idx]), 0), (int(xs[idx]), int(ys[idx])), 255, 3)

    # Нижняя граница.
    near = np.where(ys >= h - margin)[0]
    for idx in near[::max(1, len(near) // 50 or 1)]:
        cv2.line(result, (int(xs[idx]), int(ys[idx])), (int(xs[idx]), h - 1), 255, 3)

    return result


def contours_to_region_mask(
    annotated_rgb: np.ndarray,
    min_region_fraction: float = 0.0004,
    line_dilation: int = 5,
    border_width: int = 4,
) -> tuple[np.ndarray, dict]:
    """
    Возвращает псевдомаску областей, ограниченных синими линиями.
    """
    blue_line = extract_blue_line(annotated_rgb)
    blue_line = connect_line_to_border(blue_line)

    kernel = np.ones(
        (line_dilation, line_dilation),
        dtype=np.uint8,
    )
    barrier = cv2.dilate(blue_line, kernel, iterations=1)

    # Рамка помогает закрывать контуры, выходящие к краю.
    barrier[:border_width, :] = 255
    barrier[-border_width:, :] = 255
    barrier[:, :border_width] = 255
    barrier[:, -border_width:] = 255

    free_space = (barrier == 0).astype(np.uint8)

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        free_space,
        connectivity=8,
    )

    if count <= 1:
        return np.zeros_like(blue_line), {
            "regions_total": 0,
            "regions_selected": 0,
            "largest_region_fraction": 0.0,
        }

    component_areas = stats[1:, cv2.CC_STAT_AREA]
    component_ids = np.arange(1, count)

    largest_id = int(component_ids[np.argmax(component_areas)])
    min_area = annotated_rgb.shape[0] * annotated_rgb.shape[1] * min_region_fraction

    selected_ids = [
        int(component_id)
        for component_id, area in zip(component_ids, component_areas)
        if component_id != largest_id and area >= min_area
    ]

    region_mask = np.isin(labels, selected_ids).astype(np.uint8)

    # Убираем тонкую линию из маски и слегка сглаживаем.
    region_mask[barrier > 0] = 0
    region_mask = cv2.morphologyEx(
        region_mask,
        cv2.MORPH_CLOSE,
        np.ones((5, 5), dtype=np.uint8),
        iterations=1,
    )

    stats_result = {
        "regions_total": int(count - 1),
        "regions_selected": int(len(selected_ids)),
        "largest_region_fraction": float(
            stats[largest_id, cv2.CC_STAT_AREA]
            / (annotated_rgb.shape[0] * annotated_rgb.shape[1])
        ),
    }

    return region_mask, stats_result

## 4. Генерация всех масок и статистики

In [6]:
matched_pairs = pairs[
    (pairs["status"] == "matched")
    & (pairs["shape_equal"] == True)
].copy()

mask_records = []

for pair in tqdm(
    matched_pairs.itertuples(),
    total=len(matched_pairs),
    desc="Построение псевдомасок",
):
    original = load_rgb(pair.original_path)
    annotated = load_rgb(pair.annotated_path)

    mask, geometry_stats = contours_to_region_mask(annotated)
    overlay = make_overlay(original, mask)

    stem = Path(pair.filename).stem

    mask_path = TALC_MASK_DIR / f"{stem}_mask.png"
    overlay_path = TALC_OVERLAY_DIR / f"{stem}_overlay.jpg"

    save_mask(mask, mask_path)
    Image.fromarray(overlay).save(
        overlay_path,
        quality=92,
    )

    gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)

    if mask.any():
        selected_brightness = float(gray[mask > 0].mean())
    else:
        selected_brightness = np.nan

    if (mask == 0).any():
        background_brightness = float(gray[mask == 0].mean())
    else:
        background_brightness = np.nan

    talc_share = float(mask.mean() * 100)

    suspicious = (
        talc_share < 0.3
        or talc_share > 75
        or geometry_stats["regions_selected"] == 0
    )

    mask_records.append({
        "filename": pair.filename,
        "original_path": pair.original_path,
        "annotated_path": pair.annotated_path,
        "mask_path": str(mask_path.resolve()),
        "overlay_path": str(overlay_path.resolve()),
        "talc_region_share_percent": talc_share,
        "selected_brightness": selected_brightness,
        "background_brightness": background_brightness,
        "selected_minus_background_brightness": (
            selected_brightness - background_brightness
        ),
        "suspicious": suspicious,
        **geometry_stats,
    })

mask_stats = pd.DataFrame(mask_records)
mask_stats.to_csv(
    MANIFEST_DIR / "talc_mask_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    mask_stats.sort_values(
        "talc_region_share_percent",
        ascending=False,
    ).head(15)
)

print("Подозрительных масок:", int(mask_stats["suspicious"].sum()))
print("Сохранено:", MANIFEST_DIR / "talc_mask_manifest.csv")

Построение псевдомасок:   0%|          | 0/42 [00:00<?, ?it/s]

,filename,original_path,annotated_path,mask_path,overlay_path,talc_region_share_percent,selected_brightness,background_brightness,selected_minus_background_brightness,suspicious,regions_total,regions_selected,largest_region_fraction
22,DSCN4717.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,55.442378,67.714469,26.373449,41.341021,False,6,4,0.394058
24,DSCN4719.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,44.957520,51.404826,90.255573,-38.850747,False,17,14,0.445751
30,DSCN4731.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,39.152285,34.188185,95.832522,-61.644336,False,8,6,0.545973
17,DSCN3066.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,38.540685,65.124528,126.706296,-61.581768,False,14,6,0.560034
29,DSCN4730.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,36.626279,22.338839,64.380477,-42.041639,False,9,8,0.558637
33,DSCN4734.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,35.740935,37.973620,82.626089,-44.652469,False,8,7,0.581068
16,DSCN3057.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,35.311436,73.499155,108.667292,-35.168137,False,4,3,0.611356
1,2550375-1 10х.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,34.507042,82.680485,147.484094,-64.803609,False,11,10,0.594222
27,DSCN4727.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,33.784865,47.927650,114.089230,-66.161579,False,11,8,0.587828
6,2550378-1 5x.JPG,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\raw\Фото руд...,C:\Users\Мария\ore_hackathon\data\processed\ta...,C:\Users\Мария\ore_hackathon\data\processed\ta...,32.657908,29.786835,53.939454,-24.152619,False,9,6,0.585348


Подозрительных масок: 0
Сохранено: data\processed\manifests\talc_mask_manifest.csv


## 5. Страницы визуальной проверки

На каждой строке:

1. оригинал;
2. экспертная разметка;
3. автоматически заполненная область.

Синий слой должен совпадать с областями, ограниченными экспертными линиями.

In [7]:
def fit_thumbnail(
    rgb: np.ndarray,
    size: tuple[int, int],
) -> Image.Image:
    image = Image.fromarray(rgb)
    image.thumbnail(size)

    tile = Image.new("RGB", size, "white")
    x = (size[0] - image.width) // 2
    y = (size[1] - image.height) // 2
    tile.paste(image, (x, y))
    return tile


def create_review_pages(
    mask_stats: pd.DataFrame,
    rows_per_page: int = 6,
    tile_size: tuple[int, int] = (360, 270),
):
    page_width = tile_size[0] * 3
    caption_height = 35
    row_height = tile_size[1] + caption_height

    page_count = math.ceil(len(mask_stats) / rows_per_page)

    for page_index in range(page_count):
        page_rows = mask_stats.iloc[
            page_index * rows_per_page:
            (page_index + 1) * rows_per_page
        ]

        canvas = Image.new(
            "RGB",
            (
                page_width,
                row_height * len(page_rows) + 45,
            ),
            "white",
        )
        draw = ImageDraw.Draw(canvas)
        draw.text(
            (10, 12),
            f"Проверка масок талька — страница {page_index + 1}/{page_count}",
            fill="black",
        )

        for local_row, row in enumerate(page_rows.itertuples()):
            original = load_rgb(row.original_path)
            annotated = load_rgb(row.annotated_path)
            overlay = load_rgb(row.overlay_path)

            y = 45 + local_row * row_height

            for col, image_array in enumerate(
                [original, annotated, overlay]
            ):
                tile = fit_thumbnail(image_array, tile_size)
                canvas.paste(tile, (col * tile_size[0], y))

            caption = (
                f"{row.filename} | "
                f"доля={row.talc_region_share_percent:.1f}% | "
                f"regions={row.regions_selected} | "
                f"suspicious={row.suspicious}"
            )
            draw.text(
                (8, y + tile_size[1] + 7),
                caption,
                fill="red" if row.suspicious else "black",
            )

        destination = REVIEW_DIR / f"review_page_{page_index + 1:02d}.jpg"
        canvas.save(destination, quality=92)

create_review_pages(mask_stats)

print("Страницы проверки:", REVIEW_DIR.resolve())

Страницы проверки: C:\Users\Мария\ore_hackathon\data\processed\talc_review


## 6. Очищенный manifest ordinary / fine

Правила очистки:

- `talc`, размеченные версии и панорамы не входят в этот классификатор;
- при совпадении `pHash` в разных классах удаляются все конфликтующие изображения;
- при совпадении `pHash` внутри одного класса сохраняется изображение с наибольшим разрешением;
- split делается детерминированно по `sample_group`, чтобы один образец не попадал в разные выборки.

In [8]:
classifier = inventory[
    inventory["label"].isin(["ordinary", "fine"])
].copy()

classifier["pixel_count"] = (
    classifier["width"].fillna(0)
    * classifier["height"].fillna(0)
)

classifier["status"] = "candidate"
classifier["reason"] = ""

for phash, group in classifier.groupby("phash", dropna=False):
    labels = set(group["label"])

    if len(labels) > 1:
        classifier.loc[group.index, "status"] = "excluded"
        classifier.loc[group.index, "reason"] = "conflicting_phash_labels"
        continue

    if len(group) > 1:
        keep_index = (
            group.sort_values(
                ["pixel_count", "file_size_mb"],
                ascending=False,
            )
            .index[0]
        )

        drop_indexes = group.index.difference([keep_index])
        classifier.loc[drop_indexes, "status"] = "excluded"
        classifier.loc[drop_indexes, "reason"] = "same_class_duplicate"


def deterministic_split(group_name: str) -> str:
    value = int(
        hashlib.md5(
            str(group_name).encode("utf-8")
        ).hexdigest()[:8],
        16,
    ) / 0xFFFFFFFF

    if value < 0.15:
        return "validation"
    if value < 0.25:
        return "test"
    return "train"


classifier["split"] = None

included = classifier["status"] == "candidate"
classifier.loc[included, "split"] = (
    classifier.loc[included, "sample_group"]
    .fillna(classifier.loc[included, "filename"])
    .map(deterministic_split)
)

classifier.to_csv(
    MANIFEST_DIR / "classifier_manifest_all.csv",
    index=False,
    encoding="utf-8-sig",
)

classifier_clean = classifier[
    classifier["status"] == "candidate"
].copy()

classifier_clean.to_csv(
    MANIFEST_DIR / "classifier_manifest_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Исходных ordinary/fine:", len(classifier))
print("Оставлено:", len(classifier_clean))
print("Исключено:", int((classifier["status"] == "excluded").sum()))

display(
    classifier[
        classifier["status"] == "excluded"
    ]["reason"].value_counts()
)

display(
    classifier_clean.groupby(
        ["split", "label"]
    ).size().unstack(fill_value=0)
)

Исходных ordinary/fine: 1051
Оставлено: 993
Исключено: 58


reason
same_class_duplicate        32
conflicting_phash_labels    26
Name: count, dtype: int64

label,fine,ordinary
split,,
test,51,57
train,338,388
validation,70,89


## 7. Финальная сводка

После запуска пришлите:

- `data/processed/talc_review/review_page_01.jpg`;
- остальные страницы, на которых маска выглядит неверно;
- `data/processed/manifests/talc_mask_manifest.csv`;
- `data/processed/manifests/classifier_manifest_clean.csv`.

После проверки масок переходим к обучению baseline-классификатора `ordinary / fine`.

In [9]:
print("=" * 72)
print("ПОДГОТОВКА ДАННЫХ ЗАВЕРШЕНА")
print("=" * 72)

print("\nПсевдомаски талька:")
print("  Всего:", len(mask_stats))
print("  Подозрительных:", int(mask_stats["suspicious"].sum()))
print(
    "  Медианная доля отмеченной области:",
    f"{mask_stats['talc_region_share_percent'].median():.2f}%"
)

print("\nКлассификатор ordinary / fine:")
print("  Оставлено изображений:", len(classifier_clean))
print(
    classifier_clean.groupby(
        ["split", "label"]
    ).size().to_string()
)

print("\nРезультаты:")
print(PROCESSED_DIR.resolve())

ПОДГОТОВКА ДАННЫХ ЗАВЕРШЕНА

Псевдомаски талька:
  Всего: 42
  Подозрительных: 0
  Медианная доля отмеченной области: 20.82%

Классификатор ordinary / fine:
  Оставлено изображений: 993
split       label   
test        fine         51
            ordinary     57
train       fine        338
            ordinary    388
validation  fine         70
            ordinary     89

Результаты:
C:\Users\Мария\ore_hackathon\data\processed
